In [ ]:
# Daily Challenge: Text Analysis of books using word cloud

# Book URLs (Project Gutenberg)
urls = [
    'https://www.gutenberg.org/files/11/11-0.txt',  # Alice’s Adventures in Wonderland
    'https://www.gutenberg.org/files/12/12-0.txt',  # Through the Looking-Glass
    'https://www.gutenberg.org/files/19012/19012-0.txt'  # A Tangled Tale
]

import requests
import re

def load_texts(urls):
    corpus = []
    for url in urls:
        response = requests.get(url)
        text = response.text
        # Remove non-words (keep only words and spaces)
        text = re.sub(r'[^A-Za-z\s]', ' ', text)
        corpus.append(text)
    return corpus

corpus = load_texts(urls)
# Print first 200 chars of each text
for i, text in enumerate(corpus):
    print(f"Book {i+1} sample:\n", text[:200], '\n')

In [ ]:
# Remove irrelevant parts (autoral credits) using 'START' and 'END' markers
def clean_gutenberg_text(text):
    start = text.find('START')
    end = text.find('END')
    if start != -1 and end != -1:
        return text[start:end]
    return text

corpus_cleaned = [clean_gutenberg_text(t) for t in corpus]
# Print first 200 chars of cleaned text
for i, text in enumerate(corpus_cleaned):
    print(f"Book {i+1} cleaned sample:\n", text[:200], '\n')

In [ ]:
# 3. Tokenize the text and print first 150 tokens
import nltk
nltk.download('punkt')
from nltk.tokenize import word_tokenize

tokens_list = [word_tokenize(text) for text in corpus_cleaned]
for i, tokens in enumerate(tokens_list):
    print(f"Book {i+1} tokens:\n", tokens[:150], '\n')

In [ ]:
# 4. Remove stopwords using NLTK
nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

tokens_nostop = [[t for t in tokens if t.lower() not in stop_words] for tokens in tokens_list]
# Check stopwords removed
test_words = ['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves']
for i, tokens in enumerate(tokens_nostop):
    print(f"Book {i+1} stopword counts:")
    for w in test_words:
        print(f"  {w}: {tokens.count(w)}")
    print(tokens[:50], '\n')

In [ ]:
# 5. PorterStemmer: print first 50 stemmed tokens
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()
stemmed_tokens = [[stemmer.stem(t) for t in tokens] for tokens in tokens_nostop]
for i, tokens in enumerate(stemmed_tokens):
    print(f"Book {i+1} stemmed tokens:\n", tokens[:50], '\n')

In [ ]:
# 6. Lemmatization with spaCy
import spacy
nlp = spacy.load('en_core_web_sm')
lemmatized_tokens = []
for tokens in tokens_nostop:
    doc = nlp(' '.join(tokens))
    lems = [token.lemma_ for token in doc]
    lemmatized_tokens.append(lems)
for i, tokens in enumerate(lemmatized_tokens):
    print(f"Book {i+1} lemmatized tokens:\n", tokens[:50], '\n')

# 7. Analysis: Stemming vs Lemmatization
# Stemming cuts words to their root form, often producing non-words. Lemmatization returns the base dictionary form, which is usually a valid word. Lemmatization is more accurate for linguistic analysis.

In [ ]:
# 8. POS tagging with NLTK
nltk.download('averaged_perceptron_tagger')
pos_tags = [nltk.pos_tag(tokens) for tokens in tokens_nostop]
for i, tags in enumerate(pos_tags):
    print(f"Book {i+1} POS tags:\n", tags[:50], '\n')

In [ ]:
# 9. NER with NLTK (using ne_chunk)
nltk.download('maxent_ne_chunker')
nltk.download('words')
from nltk import ne_chunk
entities = [ne_chunk(tags) for tags in pos_tags]
for i, tree in enumerate(entities):
    print(f"Book {i+1} named entities (first 50):\n", tree[:50], '\n')

In [ ]:
# --- Analysis: Word Cloud ---
from wordcloud import WordCloud
import matplotlib.pyplot as plt

for i, tokens in enumerate(lemmatized_tokens):
    text = ' '.join(tokens)
    wc = WordCloud(width=800, height=400, background_color='white').generate(text)
    plt.figure(figsize=(10, 5))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title(f'Book {i+1} Word Cloud')
    plt.show()

In [ ]:
# --- BoW: Most frequent words ---
from collections import Counter

for i, tokens in enumerate(lemmatized_tokens):
    counter = Counter(tokens)
    most_common = counter.most_common(5)
    print(f"Book {i+1} 5 most frequent words:", most_common)
    # Pie plot
    labels, values = zip(*most_common)
    plt.figure(figsize=(6, 6))
    plt.pie(values, labels=[f'{l} ({v})' for l, v in most_common], autopct='%1.1f%%')
    plt.title(f'Book {i+1} Top 5 Words (BoW)')
    plt.show()

In [ ]:
# --- TF-IDF Vectorizer ---
from sklearn.feature_extraction.text import TfidfVectorizer

docs = [' '.join(tokens) for tokens in lemmatized_tokens]
tfidf = TfidfVectorizer(min_df=1, max_df=2)
X = tfidf.fit_transform(docs)

# Get top 5 words for each document by TF-IDF score
import numpy as np
for i, doc in enumerate(docs):
    row = X[i].toarray().flatten()
    top_idx = np.argsort(row)[-5:][::-1]
    features = np.array(tfidf.get_feature_names_out())[top_idx]
    scores = row[top_idx]
    print(f"Book {i+1} top 5 TF-IDF words:", list(zip(features, scores)))
    # Pie plot
    plt.figure(figsize=(6, 6))
    plt.pie(scores, labels=[f'{f} ({s:.2f})' for f, s in zip(features, scores)], autopct='%1.1f%%')
    plt.title(f'Book {i+1} Top 5 Words (TF-IDF)')
    plt.show()